# Enrichment

Enrichment adds ticket-level issue, sentiment, and emotion labels to locally stored customer-message data. It reads Preparation artifacts and writes local enriched artifacts for the Analysis phase.

---

## Libraries

Imports the standard and tabular-data libraries required by the notebook. Model-specific dependencies are added here only after the local model is selected.

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd

---

## Import

Loads the prepared ticket dataset from `enrichment/data/` for enrichment. **Run Preparation through Export before running this section.**

In [ ]:
project_root = Path.cwd()
if project_root.name == "enrichment":
    project_root = project_root.parent

prepared_tickets_path = (
    project_root / "enrichment" / "data" / "prepared_tickets.parquet"
)
if not prepared_tickets_path.is_file():
    raise FileNotFoundError(
        "Run the Preparation notebook through Export before importing data."
    )

prepared_tickets = pd.read_parquet(prepared_tickets_path)

prepared_tickets

---

## Model Selection

Selects **Qwen3 8B** as the local model for structured ticket enrichment. **Its 5.2 GB quantized size balances classification capability with the available 16 GB of unified memory.**

Ollama runs the model locally, so customer-message text remains on this machine during inference. Download the selected model with `ollama pull qwen3:8b`. See the [official Ollama documentation](https://docs.ollama.com/) for installation and local-runtime guidance.

---

## Text Profiling

Validates the structure and coverage of customer-message sequences before classification. Message collections remain separate within each ticket so enrichment can preserve issue and sentiment changes across messages.

### Message Structure

Normalizes the Parquet-loaded message collections into Python lists, then verifies their contents and counts. The source `customer_message_count` field provides an independent check on each ticket's message sequence.

In [ ]:
message_collection_is_valid = prepared_tickets["customer_messages"].map(
    lambda value: isinstance(value, (list, np.ndarray))
)
if not message_collection_is_valid.all():
    invalid_count = (~message_collection_is_valid).sum()
    raise TypeError(
        f"customer_messages contains {invalid_count} non-list-like collections."
    )

message_lists = prepared_tickets["customer_messages"].map(
    lambda value: value.tolist() if isinstance(value, np.ndarray) else value
)
message_lengths = message_lists.map(len)
flat_messages = [message for messages in message_lists for message in messages]

message_structure_profile = pd.DataFrame(
    {
        "metric": [
            "tickets",
            "message_collection_type",
            "total_messages",
            "empty_message_list_count",
            "non_string_message_count",
            "empty_or_whitespace_message_count",
            "minimum_messages_per_ticket",
            "median_messages_per_ticket",
            "maximum_messages_per_ticket",
            "source_count_mismatch_count",
        ],
        "value": [
            len(prepared_tickets),
            prepared_tickets["customer_messages"].map(
                lambda value: type(value).__name__
            ).mode().iat[0],
            len(flat_messages),
            (message_lengths == 0).sum(),
            sum(not isinstance(message, str) for message in flat_messages),
            sum(
                isinstance(message, str) and not message.strip()
                for message in flat_messages
            ),
            message_lengths.min(),
            message_lengths.median(),
            message_lengths.max(),
            (message_lengths != prepared_tickets["customer_message_count"]).sum(),
        ],
    }
)

message_structure_profile

#### *Findings*

All 10,000 ticket message collections load as NumPy arrays, each containing only strings and matching the source `customer_message_count`. There are 25,234 messages, no empty message lists or blank messages, and sequences range from 2 to 12 messages. The source provides no message-level timestamp, so later enrichment preserves source-array order as message position without interpreting it as chronological order.

---

### Message Distribution

Counts customer messages per ticket to establish the sequence lengths used by later issue and sentiment comparisons. The distribution uses tickets as its denominator.

In [ ]:
message_count_distribution = (
    message_lengths.value_counts()
    .sort_index()
    .rename_axis("message_count")
    .reset_index(name="tickets")
    .assign(ticket_share=lambda frame: frame["tickets"] / len(prepared_tickets))
)

message_count_distribution

#### *Findings*

Two-message tickets account for 67.1% of the dataset, while 95.7% contain two to four customer messages. The remaining 4.3% form a small long tail of five to 12 messages, so sequence-level comparisons remain possible without allowing longer conversations to change ticket-level denominators.